# BigAlpha 2026：稳健尾盘流动性冲击反转因子（refine39）

本 Notebook 将 `refine39.py` 整理为与已验证基线一致的正式提交格式。

核心因子以尾盘流动性冲击反转为基础，并加入：

- 聪明钱演变：三档订单簿量失衡（OBI）减挂单笔数失衡（NOI）；
- 尾盘订单簿压力；
- 尾盘相对早盘的 Amihud 冲击成本与盘口深度枯竭；
- 尾盘 VWAP 加速；
- 早盘与尾盘价格路径分歧；
- 尾盘点差和全天点差波动率质量惩罚。

提交适配：

- 数据源由评测系统动态传入 `datasources["bar1m"]`；
- 分钟数据、三档盘口和截面排名均在 BigQuant 服务端完成；
- 股票池优先使用 `bigalpha_2026_instruments`；
- 输出严格且仅包含 `date`、`instrument`、`factor` 三列；
- 不依赖本地 Feather 文件、目录路径或命令行参数。


In [ ]:
def _load_dai():
    """兼容 BigQuant AIStudio 和外部 Python 3.11 + BigQuant SDK。"""
    try:
        import dai  # BigQuant AIStudio

        return dai
    except ImportError:
        from bigquant import dai  # BigQuant SDK

        return dai


def main(datasources, start_date, end_date):
    """
    构建单个日频因子。

    Parameters
    ----------
    datasources : dict
        BigQuant 动态传入的数据表。分钟表必须来自 datasources["bar1m"]。
    start_date, end_date : str
        评估区间（含端点）。

    Returns
    -------
    pandas.DataFrame
        必须且仅含 date、instrument、factor 三列。
    """
    import numpy as np
    import pandas as pd

    dai = _load_dai()
    bar1m = datasources["bar1m"]

    # [AI-CORE]
    # 稳健尾盘流动性冲击反转：
    # 基础尾盘反转 × 聪明钱演变 × 订单簿压力
    # × 流动性枯竭 × VWAP 加速 × 路径分歧
    # × 点差质量惩罚。
    sql = f"""
    WITH min_bar AS (
        SELECT
            CAST(strftime(date, '%Y-%m-%d') AS DATETIME) AS date_day,
            instrument,
            date,
            open,
            high,
            low,
            close,
            volume,
            COALESCE(bid_volume1, 0)
                + COALESCE(bid_volume2, 0)
                + COALESCE(bid_volume3, 0) AS bid_vol13,
            COALESCE(ask_volume1, 0)
                + COALESCE(ask_volume2, 0)
                + COALESCE(ask_volume3, 0) AS ask_vol13,
            COALESCE(bid_num_orders1, 0)
                + COALESCE(bid_num_orders2, 0)
                + COALESCE(bid_num_orders3, 0) AS bid_no13,
            COALESCE(ask_num_orders1, 0)
                + COALESCE(ask_num_orders2, 0)
                + COALESCE(ask_num_orders3, 0) AS ask_no13,
            ask_price1,
            bid_price1
        FROM {bar1m}
        WHERE close > 0
          AND open > 0
    ),
    book_base AS (
        SELECT
            *,
            bid_vol13 + ask_vol13 AS depth,
            (bid_vol13 - ask_vol13)
                / (bid_vol13 + ask_vol13 + 1e-8) AS obi,
            (bid_no13 - ask_no13)
                / (bid_no13 + ask_no13 + 1e-8) AS noi,
            CASE
                WHEN ask_price1 > 0
                 AND bid_price1 > 0
                 AND ask_price1 >= bid_price1
                THEN
                    (ask_price1 - bid_price1)
                    / ((ask_price1 + bid_price1) / 2.0)
                ELSE 0.0
            END AS spread,
            abs(close / open - 1.0)
                / (volume + 1.0) AS amihud,
            close * volume AS pseudo_amount
        FROM min_bar
    ),
    work AS (
        SELECT
            *,
            obi - noi AS smart_money
        FROM book_base
    ),
    day_stats AS (
        SELECT
            date_day,
            instrument,
            first(open ORDER BY date) AS day_open,
            last(close ORDER BY date) AS day_close,
            max(high) AS day_high,
            min(low) AS day_low,
            sum(volume) AS day_volume,
            COALESCE(stddev_samp(spread), 0.0) AS day_spread_std
        FROM work
        GROUP BY date_day, instrument
    ),
    tail_stats AS (
        SELECT
            date_day,
            instrument,
            first(open ORDER BY date) AS tail_open,
            last(close ORDER BY date) AS tail_close,
            sum(volume) AS tail_volume,
            sum(pseudo_amount) AS tail_amount,
            sum(smart_money * volume) AS tail_sm,
            sum(obi * volume) AS tail_obi,
            avg(depth) AS tail_depth,
            avg(amihud) AS tail_amihud,
            avg(spread) AS tail_spread
        FROM work
        WHERE strftime(date, '%H:%M:%S') >= '14:30:00'
        GROUP BY date_day, instrument
    ),
    morning_stats AS (
        SELECT
            date_day,
            instrument,
            last(close ORDER BY date) AS morning_close,
            sum(volume) AS morning_volume,
            sum(smart_money * volume) AS morning_sm,
            avg(depth) AS morning_depth,
            avg(amihud) AS morning_amihud
        FROM work
        WHERE strftime(date, '%H:%M:%S') <= '10:00:00'
        GROUP BY date_day, instrument
    ),
    feat AS (
        SELECT
            d.date_day,
            d.instrument,
            d.day_open,
            d.day_close,
            d.day_high,
            d.day_low,
            d.day_volume,
            d.day_spread_std,
            t.tail_open,
            t.tail_close,
            t.tail_volume,
            t.tail_amount,
            t.tail_sm,
            t.tail_obi,
            COALESCE(t.tail_depth, 0.0) AS tail_depth,
            COALESCE(t.tail_amihud, 0.0) AS tail_amihud,
            COALESCE(t.tail_spread, 0.0) AS tail_spread,
            COALESCE(m.morning_close, d.day_open) AS morning_close,
            COALESCE(m.morning_volume, 0.0) AS morning_volume,
            COALESCE(m.morning_sm, 0.0) AS morning_sm,
            COALESCE(m.morning_depth, 0.0) AS morning_depth,
            COALESCE(m.morning_amihud, 0.0) AS morning_amihud
        FROM day_stats d
        JOIN tail_stats t
          ON d.date_day = t.date_day
         AND d.instrument = t.instrument
        LEFT JOIN morning_stats m
          ON d.date_day = m.date_day
         AND d.instrument = m.instrument
        WHERE t.tail_open > 0
          AND d.day_open > 0
    ),
    components AS (
        SELECT
            *,
            day_close / tail_open - 1.0 AS tail_reversal,
            tail_volume / (day_volume + 1e-8) AS tail_volume_share,
            (day_high - day_low) / day_open AS intraday_range,
            tail_sm / (tail_volume + 1e-8)
                - morning_sm / (morning_volume + 1e-8)
                AS delta_sm,
            tail_obi / (tail_volume + 1e-8) AS tail_vw_obi,
            CASE
                WHEN tail_amount / (tail_volume + 1e-8) > 0
                THEN
                    tail_close
                    / (tail_amount / (tail_volume + 1e-8))
                    - 1.0
                ELSE 0.0
            END AS vwap_dev,
            morning_close / day_open - 1.0 AS morning_ret
        FROM feat
    ),
    ranked AS (
        SELECT
            *,
            (
                rank() OVER (
                    PARTITION BY date_day
                    ORDER BY delta_sm
                )
                + (
                    count(*) OVER (
                        PARTITION BY date_day, delta_sm
                    ) - 1
                ) / 2.0
            )
            / count(*) OVER (PARTITION BY date_day)
                AS rank_delta_sm,
            (
                rank() OVER (
                    PARTITION BY date_day
                    ORDER BY tail_vw_obi
                )
                + (
                    count(*) OVER (
                        PARTITION BY date_day, tail_vw_obi
                    ) - 1
                ) / 2.0
            )
            / count(*) OVER (PARTITION BY date_day)
                AS rank_tail_vw_obi,
            (
                rank() OVER (
                    PARTITION BY date_day
                    ORDER BY tail_amihud
                )
                + (
                    count(*) OVER (
                        PARTITION BY date_day, tail_amihud
                    ) - 1
                ) / 2.0
            )
            / count(*) OVER (PARTITION BY date_day)
                AS rank_tail_amihud,
            (
                rank() OVER (
                    PARTITION BY date_day
                    ORDER BY morning_amihud
                )
                + (
                    count(*) OVER (
                        PARTITION BY date_day, morning_amihud
                    ) - 1
                ) / 2.0
            )
            / count(*) OVER (PARTITION BY date_day)
                AS rank_morning_amihud,
            (
                rank() OVER (
                    PARTITION BY date_day
                    ORDER BY tail_depth
                )
                + (
                    count(*) OVER (
                        PARTITION BY date_day, tail_depth
                    ) - 1
                ) / 2.0
            )
            / count(*) OVER (PARTITION BY date_day)
                AS rank_tail_depth,
            (
                rank() OVER (
                    PARTITION BY date_day
                    ORDER BY morning_depth
                )
                + (
                    count(*) OVER (
                        PARTITION BY date_day, morning_depth
                    ) - 1
                ) / 2.0
            )
            / count(*) OVER (PARTITION BY date_day)
                AS rank_morning_depth,
            (
                rank() OVER (
                    PARTITION BY date_day
                    ORDER BY vwap_dev
                )
                + (
                    count(*) OVER (
                        PARTITION BY date_day, vwap_dev
                    ) - 1
                ) / 2.0
            )
            / count(*) OVER (PARTITION BY date_day)
                AS rank_vwap_dev,
            (
                rank() OVER (
                    PARTITION BY date_day
                    ORDER BY tail_spread
                )
                + (
                    count(*) OVER (
                        PARTITION BY date_day, tail_spread
                    ) - 1
                ) / 2.0
            )
            / count(*) OVER (PARTITION BY date_day)
                AS rank_tail_spread,
            (
                rank() OVER (
                    PARTITION BY date_day
                    ORDER BY day_spread_std
                )
                + (
                    count(*) OVER (
                        PARTITION BY date_day, day_spread_std
                    ) - 1
                ) / 2.0
            )
            / count(*) OVER (PARTITION BY date_day)
                AS rank_day_spread_std
        FROM components
    ),
    multipliers AS (
        SELECT
            *,
            sign(-tail_reversal) AS direction,
            least(
                1.5,
                greatest(
                    0.5,
                    1.0
                    + sign(-tail_reversal)
                    * (rank_delta_sm - 0.5)
                )
            ) AS sm_multiplier,
            least(
                1.5,
                greatest(
                    0.5,
                    1.0
                    + sign(-tail_reversal)
                    * (0.5 - rank_tail_vw_obi)
                )
            ) AS pressure_multiplier,
            least(
                1.5,
                greatest(
                    0.5,
                    1.0
                    + 0.5 * (
                        rank_tail_amihud
                        - rank_morning_amihud
                    )
                    + 0.5 * (
                        rank_morning_depth
                        - rank_tail_depth
                    )
                )
            ) AS exhaustion_multiplier,
            least(
                1.5,
                greatest(
                    0.5,
                    1.0
                    + sign(-tail_reversal)
                    * (0.5 - rank_vwap_dev)
                )
            ) AS accel_multiplier,
            least(
                1.2,
                greatest(
                    0.8,
                    1.0
                    + 0.2
                    * (
                        -sign(morning_ret)
                        * sign(tail_reversal)
                    )
                )
            ) AS path_multiplier,
            1.0 - rank_tail_spread * 0.4
                AS liquidity_penalty,
            1.0 - rank_day_spread_std * 0.4
                AS volatility_penalty
        FROM ranked
    )
    SELECT
        date_day AS date,
        instrument,
        (
            -1.0
            * tail_reversal
            * tail_volume_share
            * ln(1.0 + intraday_range)
            * sm_multiplier
            * pressure_multiplier
            * exhaustion_multiplier
            * accel_multiplier
            * path_multiplier
            * liquidity_penalty
            * volatility_penalty
        ) AS factor
    FROM multipliers
    """

    raw_factor = dai.query(
        sql,
        filters={"date": [start_date, end_date]},
        compression=True,
    ).df()

    # 优先使用比赛官方股票池；异常时回退到基线股票池。
    try:
        stock_pool = dai.query(
            """
            SELECT date, instrument
            FROM bigalpha_2026_instruments
            """,
            filters={"date": [start_date, end_date]},
        ).df()
    except Exception:
        stock_pool = dai.query(
            """
            SELECT date, instrument
            FROM cn_stock_factors_base
            WHERE CAST(is_hs300 AS INTEGER) = 1
            """,
            filters={"date": [start_date, end_date]},
        ).df()

    raw_factor["date"] = pd.to_datetime(
        raw_factor["date"]
    ).dt.normalize()
    stock_pool["date"] = pd.to_datetime(
        stock_pool["date"]
    ).dt.normalize()
    raw_factor["instrument"] = raw_factor[
        "instrument"
    ].astype(str)
    stock_pool["instrument"] = stock_pool[
        "instrument"
    ].astype(str)

    raw_factor = raw_factor[
        ["date", "instrument", "factor"]
    ].drop_duplicates(
        ["date", "instrument"],
        keep="last",
    )
    stock_pool = stock_pool.drop_duplicates(
        ["date", "instrument"],
        keep="last",
    )

    factor_data = stock_pool.merge(
        raw_factor,
        how="left",
        on=["date", "instrument"],
        validate="one_to_one",
    )
    factor_data["factor"] = (
        pd.to_numeric(
            factor_data["factor"],
            errors="coerce",
        )
        .replace([np.inf, -np.inf], np.nan)
        .fillna(0.0)
        .astype(float)
    )

    start_day = pd.to_datetime(start_date).normalize()
    end_day = pd.to_datetime(end_date).normalize()
    factor_data = factor_data[
        (factor_data["date"] >= start_day)
        & (factor_data["date"] <= end_day)
    ]

    factor_data = (
        factor_data[
            ["date", "instrument", "factor"]
        ]
        .sort_values(["date", "instrument"])
        .reset_index(drop=True)
    )

    if factor_data.empty:
        raise ValueError("No factor rows were generated.")
    if factor_data.duplicated(
        ["date", "instrument"]
    ).any():
        raise ValueError(
            "Duplicate date-instrument rows detected."
        )
    if not np.isfinite(
        factor_data["factor"].to_numpy(dtype=float)
    ).all():
        raise ValueError(
            "Non-finite factor values detected."
        )
    if list(factor_data.columns) != [
        "date",
        "instrument",
        "factor",
    ]:
        raise ValueError(
            "Output columns must be date, instrument, factor."
        )

    return factor_data
